# sparse_release_drawdown_budget_guard_screen_v1

Notebook-first screen for the next mechanism after `sparse_release_failed_fold_interval_diagnostic_v1`.

The prior diagnostic showed that the selected PPO checkpoint improved validation and `wf_2022` relative return, but widened `wf_2022` MaxDD because it held more exposure through the 2022-03-30 .. 2022-10-14 drawdown path. This notebook tests whether a path-dependent drawdown budget can safely sit between PPO output and execution.

Governance fields:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `sparse_release_bc_ppo_screen_v1` and its failed-fold diagnostic.
- **Role**: `diagnose`.
- **Changed layer**: `execution` / `model_selection` risk gate screen only; no actor promotion.
- **Target windows**: `wf_2022` selected-PPO failed fold and `wf_2023_2024` selected-BC fold.
- **Accepted trade-off**: a risk gate must keep `relative_return >= 30pp` and `MaxDD <= 15%` on both windows without collapsing into a near-cash trivial solution.
- **Failure mode**: gate only fixes 2022 by flattening exposure, or it degrades 2023/2024 participation, so the next step must move into checkpoint objective / state-conditioned training rather than execution-only caps.

External basis:

- Risk-sensitive MDP / CVaR: https://arxiv.org/abs/1506.02188
- Deep portfolio RL with explicit portfolio reallocation: https://arxiv.org/abs/1706.10059
- Qlib reproducible research harness: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL reproducible financial RL framework: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Probability of Backtest Overfitting / CSCV: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import copy
import json
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from src.soxl_quant.app.drawdown_budget_overlay import (
    DrawdownBudgetOverlayConfig,
    apply_drawdown_budget_overlay,
    drawdown_budget_overlay_metrics,
    screen_drawdown_budget_overlay_configs,
)
from src.soxl_quant.app.path_diagnostics import make_backtest_daily_frame
from src.soxl_quant.app.state_objective_bc import StateObjectiveBCTrainingConfig, fit_state_objective_bc_actor_from_frame
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.environment import build_price_path
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'sparse_release_drawdown_budget_guard_screen_v1'
SOURCE_EXPERIMENT = 'sparse_release_failed_fold_interval_diagnostic_v1'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime/model_selection/card' / SOURCE_EXPERIMENT
TARGET_CARD = ROOT / 'runtime/model_selection/card/sparse_rebound_release_target_revision_v1'
TARGETS_PATH = TARGET_CARD / 'sparse_release_revised_targets.csv'
FRAME_PATH = ROOT / 'data/soxl_profile_balanced_rule_residual_v22.pkl'
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
MIN_AVG_POS_SANITY = 0.10
CARD.mkdir(parents=True, exist_ok=True)

WINDOWS = {
    'wf_2022': {
        'source_path': SOURCE_CARD / 'ppo_selected_daily_2022.csv',
        'comparison_path': SOURCE_CARD / 'bc_daily_2022.csv',
        'selected_path_label': 'ppo_selected',
        'test': ('2022-01-01', '2022-12-31'),
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
    },
    'wf_2023_2024': {
        'selected_path_label': 'bc_selected',
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}

CONFIGS = [
    ('no_guard', DrawdownBudgetOverlayConfig(enabled=False)),
    ('cvar_budget_10_15_cash', DrawdownBudgetOverlayConfig(start_drawdown=0.10, full_drawdown=0.15, min_position=0.0, trade_threshold=0.0, trade_lock_days=0, sell_trade_threshold=0.0, sell_trade_lock_days=0)),
    ('cvar_budget_08_15_floor05', DrawdownBudgetOverlayConfig(start_drawdown=0.08, full_drawdown=0.15, min_position=0.05, trade_threshold=0.0, trade_lock_days=0, sell_trade_threshold=0.0, sell_trade_lock_days=0)),
    ('cvar_budget_10_20_floor10', DrawdownBudgetOverlayConfig(start_drawdown=0.10, full_drawdown=0.20, min_position=0.10, trade_threshold=0.0, trade_lock_days=0, sell_trade_threshold=0.0, sell_trade_lock_days=0)),
    ('cvar_budget_12_25_floor15', DrawdownBudgetOverlayConfig(start_drawdown=0.12, full_drawdown=0.25, min_position=0.15, trade_threshold=0.0, trade_lock_days=0, sell_trade_threshold=0.0, sell_trade_lock_days=0)),
]


def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def to_json_safe(value):
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [to_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(to_json_safe(payload), indent=2, sort_keys=True, default=json_default, allow_nan=False) + '\n', encoding='utf-8')


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
profile = load_profile(PROFILE_NAME)
source_summary = json.loads((SOURCE_CARD / 'summary.json').read_text(encoding='utf-8'))
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
frame = pd.read_pickle(FRAME_PATH)
coverage = {
    'experiment': EXPERIMENT,
    'source_experiment': SOURCE_EXPERIMENT,
    'source_decision': source_summary.get('decision'),
    'profile': PROFILE_NAME,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'min_avg_pos_sanity': MIN_AVG_POS_SANITY,
    'configs': [{'name': name, **cfg.__dict__} for name, cfg in CONFIGS],
    'windows': {name: {k: str(v) for k, v in spec.items() if k not in {'source_path', 'comparison_path'}} for name, spec in WINDOWS.items()},
}
write_json(CARD / 'coverage.json', coverage)
pd.DataFrame(coverage['configs']).to_csv(CARD / 'config_grid.csv', index=False)
coverage


{'experiment': 'sparse_release_drawdown_budget_guard_screen_v1',
 'source_experiment': 'sparse_release_failed_fold_interval_diagnostic_v1',
 'source_decision': 'ppo_failed_wf_2022_drawdown_interval_no_longer_training',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 15.0,
 'min_avg_pos_sanity': 0.1,
 'configs': [{'name': 'no_guard',
   'enabled': False,
   'start_drawdown': 0.1,
   'full_drawdown': 0.2,
   'min_position': 0.25,
   'trade_threshold': 0.0,
   'trade_lock_days': 0,
   'sell_trade_threshold': 0.0,
   'sell_trade_lock_days': 0},
  {'name': 'cvar_budget_10_15_cash',
   'enabled': True,
   'start_drawdown': 0.1,
   'full_drawdown': 0.15,
   'min_position': 0.0,
   'trade_threshold': 0.0,
   'trade_lock_days': 0,
   'sell_trade_threshold': 0.0,
   'sell_trade_lock_days': 0},
  {'name': 'cvar_budget_08_15_floor05',
   'enabled': True,
   'start_drawdown': 0.08,
   'full_drawdown': 0.15,
   'min_position': 0.05,
   'trade_

In [3]:
def price_for_window(profile, spec: dict) -> pd.Series:
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    test_df = segment_frame(dataset, 'test')
    return pd.Series(build_price_path(test_df), index=test_df.index, name='price')


def rebuild_wf_2024_selected_daily() -> tuple[pd.DataFrame, pd.Series]:
    spec = WINDOWS['wf_2023_2024']
    train_end = pd.Timestamp(spec['train'][1])
    fold_targets = targets[(targets['fold'] == spec['target_fold']) & (targets['datetime'] <= train_end)].copy()
    set_seed(SEED)
    bc_fit = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        train_config=StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED),
        sample_weight_config=None,
        sample_size=0,
    )
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    test_df = segment_frame(dataset, 'test')
    price = pd.Series(build_price_path(test_df), index=test_df.index, name='price')
    _, actor = build_policy(observation_dim_for_profile(profile), profile)
    actor.load_state_dict(copy.deepcopy(bc_fit.actor.state_dict()))
    actor.eval()
    result = backtest_profile(
        actor,
        test_df,
        price,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns_for_packs(profile.feature.packs, profile.name),
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=spec['test'],
        trace_decisions=True,
    )
    daily = make_backtest_daily_frame(result, label='wf_2023_2024_bc_selected', feature_frame=test_df)
    daily.to_csv(CARD / 'wf_2023_2024_bc_selected_daily.csv', index=False)
    return daily, price


def active_replay_inputs(daily: pd.DataFrame, price: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    frame = daily.copy()
    frame['datetime'] = pd.to_datetime(frame['datetime'])
    changed = frame['account'].astype(float).diff().abs().fillna(0.0) > 1e-9 if 'account' in frame.columns else pd.Series(True, index=frame.index)
    if changed.any():
        start_idx = max(int(np.flatnonzero(changed.to_numpy())[0]) - 1, 0)
    else:
        start_idx = 0
    frame = frame.iloc[start_idx:].reset_index(drop=True)
    start = pd.Timestamp(frame['datetime'].min())
    end = pd.Timestamp(frame['datetime'].max())
    px = price.copy()
    px.index = pd.to_datetime(px.index)
    px = px.loc[(px.index >= start) & (px.index <= end)]
    return frame, px


In [4]:
init_qlib(QLIB_DATA_PATH)
window_daily: dict[str, pd.DataFrame] = {}
window_price: dict[str, pd.Series] = {}

wf2022_raw = pd.read_csv(WINDOWS['wf_2022']['source_path'], parse_dates=['datetime'])
wf2022_price_raw = price_for_window(profile, WINDOWS['wf_2022'])
wf2022, wf2022_price = active_replay_inputs(wf2022_raw, wf2022_price_raw)
window_daily['wf_2022'] = wf2022
window_price['wf_2022'] = wf2022_price

wf2024_raw, price2024_raw = rebuild_wf_2024_selected_daily()
wf2024, price2024 = active_replay_inputs(wf2024_raw, price2024_raw)
window_daily['wf_2023_2024'] = wf2024
window_price['wf_2023_2024'] = price2024

pd.DataFrame([
    {'window': name, 'rows': len(daily), 'start': str(pd.to_datetime(daily['datetime']).min().date()), 'end': str(pd.to_datetime(daily['datetime']).max().date())}
    for name, daily in window_daily.items()
]).to_csv(CARD / 'window_coverage.csv', index=False)
window_daily['wf_2022'].head(), window_daily['wf_2023_2024'].head()


[82822:MainThread](2026-05-26 03:40:54,151) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[82822:MainThread](2026-05-26 03:40:54,861) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[82822:MainThread](2026-05-26 03:40:54,861) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[82822:MainThread](2026-05-26 03:40:57,129) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[82822:MainThread](2026-05-26 03:40:57,132) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

(    datetime         label       account  position       nav  daily_return  \
 0 2022-03-29  ppo_selected  1.000000e+06  0.278275  1.000000      0.000000   
 1 2022-03-30  ppo_selected  1.015187e+06  0.250000  1.015187      0.015187   
 2 2022-03-31  ppo_selected  9.997673e+05  0.234169  0.999767     -0.015189   
 3 2022-04-01  ppo_selected  9.848273e+05  0.221523  0.984827     -0.014943   
 4 2022-04-04  ppo_selected  9.948473e+05  0.215754  0.994847      0.010174   
 
        peak  drawdown    return  anchor_gap_180  ...    mom_10  \
 0  1.000000  0.000000  0.067197       -0.064376  ...  0.396679   
 1  1.015187  0.000000 -0.092804       -0.151127  ...  0.151859   
 2  1.015187  0.015189 -0.062152       -0.203829  ...  0.071801   
 3  1.015187  0.029906 -0.064208       -0.254910  ... -0.052489   
 4  1.015187  0.020036  0.046018       -0.220721  ... -0.002131   
 
    anchor_gap_velocity_10  trend_gap_21_velocity  rsi_14_velocity_5  \
 0                0.290784               0.08250

In [5]:
def evaluate_configs_for_window(window: str, daily: pd.DataFrame, price: pd.Series) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    rows = []
    overlays = {}
    base = daily[['datetime', 'position']].copy()
    for candidate_id, (name, cfg) in enumerate(CONFIGS):
        overlay = apply_drawdown_budget_overlay(base, price, cfg, position_col='position')
        metrics = drawdown_budget_overlay_metrics(
            overlay,
            price,
            target_relative_return=TARGET_RELATIVE_RETURN,
            target_max_dd=TARGET_MAX_DD,
        )
        metrics['avg_pos_sanity'] = bool(float(metrics['avg_pos']) >= MIN_AVG_POS_SANITY or name == 'no_guard')
        metrics['window_gate'] = bool(metrics['target_gate'] and metrics['avg_pos_sanity'])
        row = {
            'window': window,
            'candidate_id': candidate_id,
            'config_name': name,
            **cfg.__dict__,
            **metrics,
        }
        rows.append(row)
        out = overlay.copy()
        out['window'] = window
        out['config_name'] = name
        overlays[name] = out
    return pd.DataFrame(rows), overlays

metric_frames = []
overlay_paths: dict[tuple[str, str], pd.DataFrame] = {}
for window, daily in window_daily.items():
    metrics, overlays = evaluate_configs_for_window(window, daily, window_price[window])
    metric_frames.append(metrics)
    for name, overlay in overlays.items():
        overlay_paths[(window, name)] = overlay
        overlay.to_csv(CARD / f'{window}_{name}_daily.csv', index=False)

window_metrics = pd.concat(metric_frames, ignore_index=True)
window_metrics.to_csv(CARD / 'window_guard_metrics.csv', index=False)

aggregate_rows = []
for name, _ in CONFIGS:
    sub = window_metrics[window_metrics['config_name'].eq(name)]
    aggregate_rows.append({
        'config_name': name,
        'windows': int(len(sub)),
        'full_gate': bool(len(sub) == len(WINDOWS) and sub['window_gate'].all()),
        'target_gate_count': int(sub['target_gate'].sum()),
        'avg_pos_sanity_count': int(sub['avg_pos_sanity'].sum()),
        'max_drawdown_gap': float(sub['drawdown_gap'].max()),
        'max_relative_gap': float(sub['relative_gap'].max()),
        'mean_avg_pos': float(sub['avg_pos'].mean()),
        'mean_robust_score': float(sub['robust_score'].mean()),
        'total_changed_days': int(sub['changed_days'].sum()),
        'worst_window': str(sub.sort_values(['drawdown_gap', 'relative_gap'], ascending=[False, False]).iloc[0]['window']) if len(sub) else '',
    })
aggregate = pd.DataFrame(aggregate_rows).sort_values(['full_gate', 'max_drawdown_gap', 'max_relative_gap', 'mean_robust_score'], ascending=[False, True, True, True]).reset_index(drop=True)
aggregate.to_csv(CARD / 'aggregate_guard_metrics.csv', index=False)

display(window_metrics)
display(aggregate)


,window,candidate_id,config_name,enabled,start_drawdown,full_drawdown,min_position,trade_threshold,trade_lock_days,sell_trade_threshold,...,avg_pos,turnover,active_days,changed_days,relative_gap,drawdown_gap,target_gate,robust_score,avg_pos_sanity,window_gate
0,wf_2022,0,no_guard,False,0.10,0.20,0.25,0.0,0,0.0,...,0.367058,2.699467,0,0,0.000000,31.523521,False,3974.983583,True,False
1,wf_2022,1,cvar_budget_10_15_cash,True,0.10,0.15,0.00,0.0,0,0.0,...,0.024991,0.550001,179,172,0.000000,0.457891,False,0.849656,False,False
2,wf_2022,2,cvar_budget_08_15_floor05,True,0.08,0.15,0.05,0.0,0,0.0,...,0.070266,0.647376,185,173,0.000000,4.577492,False,83.826669,False,False
3,wf_2022,3,cvar_budget_10_20_floor10,True,0.10,0.20,0.10,0.0,0,0.0,...,0.128684,1.413087,178,163,0.000000,12.949977,False,670.835862,True,False
4,wf_2022,4,cvar_budget_12_25_floor15,True,0.12,0.25,0.15,0.0,0,0.0,...,0.217552,2.390405,167,124,0.000000,19.665812,False,1547.024400,True,False
5,wf_2023_2024,0,no_guard,False,0.10,0.20,0.25,0.0,0,0.0,...,0.151048,2.070461,0,0,1.752829,0.996730,False,7.087700,True,False
6,wf_2023_2024,1,cvar_budget_10_15_cash,True,0.10,0.15,0.00,0.0,0,0.0,...,0.077339,1.720001,109,81,4.007458,0.786990,False,18.571532,False,False
7,wf_2023_2024,2,cvar_budget_08_15_floor05,True,0.08,0.15,0.05,0.0,0,0.0,...,0.101588,2.423174,128,81,5.024431,2.142734,False,43.658601,True,False
8,wf_2023_2024,3,cvar_budget_10_20_floor10,True,0.10,0.20,0.10,0.0,0,0.0,...,0.151048,2.070461,102,0,1.752829,0.996730,False,7.087700,True,False
9,wf_2023_2024,4,cvar_budget_12_25_floor15,True,0.12,0.25,0.15,0.0,0,0.0,...,0.151048,2.070461,58,0,1.752829,0.996730,False,7.087700,True,False


,config_name,windows,full_gate,target_gate_count,avg_pos_sanity_count,max_drawdown_gap,max_relative_gap,mean_avg_pos,mean_robust_score,total_changed_days,worst_window
0,cvar_budget_10_15_cash,2,False,0,0,0.786990,4.007458,0.051165,9.710594,253,wf_2023_2024
1,cvar_budget_08_15_floor05,2,False,0,1,4.577492,5.024431,0.085927,63.742635,254,wf_2022
2,cvar_budget_10_20_floor10,2,False,0,2,12.949977,1.752829,0.139866,338.961781,163,wf_2022
3,cvar_budget_12_25_floor15,2,False,0,2,19.665812,1.752829,0.184300,777.056050,124,wf_2022
4,no_guard,2,False,0,2,31.523521,1.752829,0.259053,1991.035641,0,wf_2022


In [6]:
# Also run the reusable train-window screen API on the 2022 failed path to keep the shared interface covered by a real artifact.
wf2022_screen = screen_drawdown_budget_overlay_configs(
    window_daily['wf_2022'][['datetime', 'position']],
    window_price['wf_2022'],
    [cfg for _, cfg in CONFIGS],
    fold='wf_2022_selected_path_full_window_diagnostic',
    train_start=WINDOWS['wf_2022']['test'][0],
    train_end=WINDOWS['wf_2022']['test'][1],
    target_relative_return=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
)
wf2022_screen.insert(2, 'config_name', [CONFIGS[int(idx)][0] for idx in wf2022_screen['candidate_id']])
wf2022_screen.to_csv(CARD / 'wf_2022_reusable_screen_api.csv', index=False)
display(wf2022_screen)


,fold,candidate_id,config_name,enabled,start_drawdown,full_drawdown,min_position,trade_threshold,trade_lock_days,sell_trade_threshold,...,train_avg_pos,train_turnover,train_active_days,train_changed_days,train_relative_gap,train_drawdown_gap,train_target_gate,train_robust_score,target_gate,robust_score
0,wf_2022_selected_path_full_window_diagnostic,1,cvar_budget_10_15_cash,True,0.10,0.15,0.00,0.0,0,0.0,...,0.024991,0.550001,179,172,0.0,0.457891,False,0.849656,False,0.849656
1,wf_2022_selected_path_full_window_diagnostic,2,cvar_budget_08_15_floor05,True,0.08,0.15,0.05,0.0,0,0.0,...,0.070266,0.647376,185,173,0.0,4.577492,False,83.826669,False,83.826669
2,wf_2022_selected_path_full_window_diagnostic,3,cvar_budget_10_20_floor10,True,0.10,0.20,0.10,0.0,0,0.0,...,0.128684,1.413087,178,163,0.0,12.949977,False,670.835862,False,670.835862
3,wf_2022_selected_path_full_window_diagnostic,4,cvar_budget_12_25_floor15,True,0.12,0.25,0.15,0.0,0,0.0,...,0.217552,2.390405,167,124,0.0,19.665812,False,1547.024400,False,1547.024400
4,wf_2022_selected_path_full_window_diagnostic,0,no_guard,False,0.10,0.20,0.25,0.0,0,0.0,...,0.367058,2.699467,0,0,0.0,31.523521,False,3974.983583,False,3974.983583


In [7]:
best = aggregate.iloc[0].to_dict() if not aggregate.empty else {}
full_gate = bool(best.get('full_gate', False)) if best else False
near_cash_configs = aggregate[aggregate['mean_avg_pos'] < MIN_AVG_POS_SANITY]['config_name'].tolist() if not aggregate.empty else []
metrics_by_config = {
    name: window_metrics[window_metrics['config_name'].eq(name)].to_dict(orient='records')
    for name, _ in CONFIGS
}

conclusion = []
if best:
    conclusion.append(
        f"Best aggregate config `{best['config_name']}` full_gate={bool(best['full_gate'])}, "
        f"max_drawdown_gap={best['max_drawdown_gap']:.2f}pp, max_relative_gap={best['max_relative_gap']:.2f}pp, "
        f"mean_avg_pos={best['mean_avg_pos']:.3f}."
    )
for name in [cfg_name for cfg_name, _ in CONFIGS]:
    sub = window_metrics[window_metrics['config_name'].eq(name)]
    parts = []
    for row in sub.to_dict(orient='records'):
        parts.append(f"{row['window']}: rel={row['relative_return']:.2f}pp/dd={row['max_dd']:.2f}%/avg={row['avg_pos']:.3f}/gate={bool(row['window_gate'])}")
    conclusion.append(f"{name}: " + '; '.join(parts))
if near_cash_configs:
    conclusion.append(
        'Near-cash warning: configs below avg-position sanity are diagnostic only, not promotion candidates: '
        + ', '.join(near_cash_configs)
        + '.'
    )
if full_gate:
    decision = 'drawdown_budget_guard_candidate_needs_full_backtest_replay'
    next_mechanism = 'integrate_guard_into_backtest_replay_and_validate_full_walkforward'
    promotion_allowed = False
    conclusion.append('Decision: no promotion from this notebook; rerun through canonical qlib backtest/replay before considering runtime integration.')
else:
    decision = 'drawdown_budget_guard_screen_no_promotion'
    next_mechanism = 'move_drawdown_control_into_checkpoint_objective_or_state_conditioned_training'
    promotion_allowed = False
    conclusion.append('Decision: execution-only drawdown budget is not enough for promotion; next step should move drawdown control into checkpoint selection/training objective.')

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'source_experiment': SOURCE_EXPERIMENT,
    'decision': decision,
    'next_mechanism': next_mechanism,
    'promotion_allowed': promotion_allowed,
    'profile': PROFILE_NAME,
    'role': 'diagnose',
    'changed_layer': ['execution', 'model_selection'],
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'min_avg_pos_sanity': MIN_AVG_POS_SANITY,
    'best_config': best,
    'external_sources': [
        'https://arxiv.org/abs/1506.02188',
        'https://arxiv.org/abs/1706.10059',
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
    'sidecars': [
        'coverage.json',
        'config_grid.csv',
        'window_coverage.csv',
        'wf_2023_2024_bc_selected_daily.csv',
        'window_guard_metrics.csv',
        'aggregate_guard_metrics.csv',
        'wf_2022_reusable_screen_api.csv',
        'selection.json',
        'summary.json',
    ],
}
for window, _ in WINDOWS.items():
    for name, _ in CONFIGS:
        selection['sidecars'].append(f'{window}_{name}_daily.csv')
summary = {
    'experiment': EXPERIMENT,
    'source_experiment': SOURCE_EXPERIMENT,
    'decision': decision,
    'coverage': coverage,
    'window_metrics': metrics_by_config,
    'aggregate': aggregate.to_dict(orient='records'),
    'conclusion': conclusion,
    'selection': selection,
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
for line in conclusion:
    display(Markdown(f'- {line}'))
selection


- Best aggregate config `cvar_budget_10_15_cash` full_gate=False, max_drawdown_gap=0.79pp, max_relative_gap=4.01pp, mean_avg_pos=0.051.

- no_guard: wf_2022: rel=42.19pp/dd=46.52%/avg=0.367/gate=False; wf_2023_2024: rel=28.25pp/dd=16.00%/avg=0.151/gate=False

- cvar_budget_10_15_cash: wf_2022: rel=63.33pp/dd=15.46%/avg=0.025/gate=False; wf_2023_2024: rel=25.99pp/dd=15.79%/avg=0.077/gate=False

- cvar_budget_08_15_floor05: wf_2022: rel=61.18pp/dd=19.58%/avg=0.070/gate=False; wf_2023_2024: rel=24.98pp/dd=17.14%/avg=0.102/gate=False

- cvar_budget_10_20_floor10: wf_2022: rel=54.35pp/dd=27.95%/avg=0.129/gate=False; wf_2023_2024: rel=28.25pp/dd=16.00%/avg=0.151/gate=False

- cvar_budget_12_25_floor15: wf_2022: rel=48.88pp/dd=34.67%/avg=0.218/gate=False; wf_2023_2024: rel=28.25pp/dd=16.00%/avg=0.151/gate=False

- Near-cash warning: configs below avg-position sanity are diagnostic only, not promotion candidates: cvar_budget_10_15_cash, cvar_budget_08_15_floor05.

- Decision: execution-only drawdown budget is not enough for promotion; next step should move drawdown control into checkpoint selection/training objective.

{'experiment': 'sparse_release_drawdown_budget_guard_screen_v1',
 'created_at': '2026-05-25T19:41:12.184712+00:00',
 'source_experiment': 'sparse_release_failed_fold_interval_diagnostic_v1',
 'decision': 'drawdown_budget_guard_screen_no_promotion',
 'next_mechanism': 'move_drawdown_control_into_checkpoint_objective_or_state_conditioned_training',
 'promotion_allowed': False,
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'role': 'diagnose',
 'changed_layer': ['execution', 'model_selection'],
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 15.0,
 'min_avg_pos_sanity': 0.1,
 'best_config': {'config_name': 'cvar_budget_10_15_cash',
  'windows': 2,
  'full_gate': False,
  'target_gate_count': 0,
  'avg_pos_sanity_count': 0,
  'max_drawdown_gap': 0.7869898207450561,
  'max_relative_gap': 4.007458010291202,
  'mean_avg_pos': 0.05116526449456248,
  'mean_robust_score': 9.710593931060211,
  'total_changed_days': 253,
  'worst_window': 'wf_2023_2024'},
 'external_sources': ['https://